In [1]:
pip install scipy statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from scipy.stats import levene
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway

from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [3]:
CLEANED_PATH = r"C:\Users\saran\guvi project\cart2insights\data\cleaned"

orders = pd.read_csv(
    CLEANED_PATH + r"\orders_cleaned.csv"
)

reviews = pd.read_csv(
    CLEANED_PATH + r"\order_reviews_cleaned.csv"
)

In [4]:
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

In [5]:
orders["delivery_status"] = np.nan

valid = (
    orders["order_delivered_customer_date"].notna()
    &
    orders["order_estimated_delivery_date"].notna()
)

orders.loc[
    valid &
    (
        orders["order_delivered_customer_date"]
        <= orders["order_estimated_delivery_date"]
    ),
    "delivery_status"
] = "On Time"

orders.loc[
    valid &
    (
        orders["order_delivered_customer_date"]
        > orders["order_estimated_delivery_date"]
    ),
    "delivery_status"
] = "Delayed"

C:\Users\saran\AppData\Local\Temp\ipykernel_1056\3084923695.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'On Time' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  orders.loc[


In [6]:
review_delivery = reviews.merge(
    orders[["order_id", "delivery_status"]],
    on="order_id",
    how="inner"
)

review_delivery = review_delivery[
    review_delivery["delivery_status"].isin(
        ["On Time", "Delayed"]
    )
]

display(
    review_delivery.groupby("delivery_status")["review_score"]
    .agg(["count", "mean", "std"])
)

,count,mean,std
delivery_status,,,
Delayed,7701,2.566550,1.658240
On Time,88658,4.293578,1.147938


In [7]:
on_time_reviews = review_delivery.loc[
    review_delivery["delivery_status"] == "On Time",
    "review_score"
].dropna()

delayed_reviews = review_delivery.loc[
    review_delivery["delivery_status"] == "Delayed",
    "review_score"
].dropna()

t_stat, p_value = ttest_ind(
    on_time_reviews,
    delayed_reviews,
    equal_var=False
)

print("Welch t-statistic:", t_stat)
print("p-value:", p_value)

Welch t-statistic: 89.55081413351807
p-value: 0.0


In [8]:
if p_value < 0.05:
    print(
        "There is a statistically significant difference "
        "in review scores between on-time and delayed deliveries."
    )
else:
    print(
        "There is no statistically significant difference "
        "in review scores."
    )

There is a statistically significant difference in review scores between on-time and delayed deliveries.


In [9]:
order_items = pd.read_csv(
    CLEANED_PATH + r"\order_items_cleaned.csv"
)

products = pd.read_csv(
    CLEANED_PATH + r"\products_cleaned.csv"
)

translation = pd.read_csv(
    CLEANED_PATH + r"\product_category_name_translation_cleaned.csv"
)

In [10]:
items = order_items.merge(
    products[[
        "product_id",
        "product_category_name"
    ]],
    on="product_id",
    how="left"
)

items = items.merge(
    translation,
    on="product_category_name",
    how="left"
)

items["category"] = items[
    "product_category_name_english"
].fillna(
    items["product_category_name"]
).fillna("unknown")

In [11]:
items["order_value"] = (
    items["price"] +
    items["freight_value"]
)

In [13]:
category_value = (
    items
    .groupby(
        ["order_id", "category"],
        as_index=False
    )["price"]
    .sum()
)

primary_category = (
    category_value
    .sort_values(
        ["order_id", "price"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
)

primary_category = primary_category[
    ["order_id", "category"]
]

print("Primary category created successfully!")
print(primary_category.head())

Primary category created successfully!
                           order_id         category
0  00010242fe8c5a6d1ba2dd792cb16214       cool_stuff
1  00018f77f2f0320c557190d7a144bdd3         pet_shop
2  000229ec398224ef6ca0657da4fc703e  furniture_decor
3  00024acbcdf0a6daa1e931b038114c75        perfumery
4  00042b26cf59d7ce69dfabb4e55b4fd9     garden_tools


In [14]:
order_values = (
    items
    .groupby("order_id", as_index=False)
    .agg(
        order_value=("order_value", "sum")
    )
)

h2_data = order_values.merge(
    primary_category,
    on="order_id",
    how="inner"
)

display(
    h2_data.groupby("category")["order_value"]
    .agg(["count", "mean", "std"])
    .sort_values("mean", ascending=False)
)

,count,mean,std
category,,,
computers,181,1286.948011,737.143982
small_appliances_home_oven_and_coffee,75,669.247600,629.511844
home_appliances_2,234,529.747521,559.760451
agro_industry_and_commerce,182,430.626758,616.049081
musical_instruments,624,336.697404,485.185889
...,...,...,...
food,444,81.647748,51.169325
cds_dvds_musicals,12,79.582500,25.944240
diapers_and_hygiene,27,79.306296,46.828592


In [15]:
category_counts = (
    h2_data["category"]
    .value_counts()
)

valid_categories = category_counts[
    category_counts >= 30
].index

h2_test = h2_data[
    h2_data["category"].isin(valid_categories)
]

In [16]:
groups = [
    group["order_value"].values
    for _, group in h2_test.groupby("category")
]

f_stat, p_value = f_oneway(*groups)

print("ANOVA F-statistic:", f_stat)
print("p-value:", p_value)

ANOVA F-statistic: 173.5729452830601
p-value: 0.0


In [17]:
if p_value < 0.05:
    print(
        "There is a statistically significant difference "
        "in average order value across product categories."
    )
else:
    print(
        "There is no statistically significant difference "
        "in average order value across product categories."
    )

There is a statistically significant difference in average order value across product categories.


In [18]:
tukey = pairwise_tukeyhsd(
    endog=h2_test["order_value"],
    groups=h2_test["category"],
    alpha=0.05
)

print(tukey)

                                     Multiple Comparison of Means - Tukey HSD, FWER=0.05                                      
                 group1                                  group2                  meandiff  p-adj    lower      upper    reject
------------------------------------------------------------------------------------------------------------------------------
             agro_industry_and_commerce                        air_conditioning  -185.4402    0.0  -268.9542  -101.9262   True
             agro_industry_and_commerce                                     art  -288.9926    0.0  -377.0469  -200.9383   True
             agro_industry_and_commerce                                   audio  -269.4359    0.0  -348.0488  -190.8231   True
             agro_industry_and_commerce                                    auto  -253.9004    0.0  -319.0139  -188.7869   True
             agro_industry_and_commerce                                    baby  -261.1203    0.0  -326.7703  -

In [19]:
payments = pd.read_csv(
    CLEANED_PATH + r"\order_payments_cleaned.csv"
)

orders = pd.read_csv(
    CLEANED_PATH + r"\orders_cleaned.csv"
)

In [20]:
payment_order = (
    payments
    .sort_values("payment_sequential")
    .drop_duplicates("order_id")
)

h3_data = payment_order.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="inner"
)

In [21]:
contingency_table = pd.crosstab(
    h3_data["payment_type"],
    h3_data["order_status"]
)

display(contingency_table)

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
payment_type,,,,,,,,
boleto,0,95,2,19191,67,70,209,150
credit_card,2,444,3,74303,239,222,848,443
debit_card,0,7,0,1485,6,2,22,6
not_defined,0,3,0,0,0,0,0,0
voucher,0,76,0,1498,2,7,28,10


In [22]:
chi2, p_value, dof, expected = chi2_contingency(
    contingency_table
)

print("Chi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p_value)

Chi-square statistic: 939.509784002784
Degrees of freedom: 28
p-value: 8.714660119444278e-180


In [23]:
if p_value < 0.05:
    print(
        "There is a statistically significant association "
        "between payment method and order status."
    )
else:
    print(
        "There is no statistically significant association "
        "between payment method and order status."
    )

There is a statistically significant association between payment method and order status.


In [24]:
groups = [
    group["order_value"].values
    for _, group in h2_test.groupby("category")
]

h2_f_stat, h2_p_value = f_oneway(*groups)

print("H2 ANOVA F-statistic:", h2_f_stat)
print("H2 ANOVA p-value:", h2_p_value)

H2 ANOVA F-statistic: 173.5729452830601
H2 ANOVA p-value: 0.0


In [25]:
if h2_p_value < 0.05:
    print(
        "There is a statistically significant difference "
        "in average order value across product categories."
    )
else:
    print(
        "There is no statistically significant difference "
        "in average order value across product categories."
    )

There is a statistically significant difference in average order value across product categories.
